# 2.5D Foveated Grid Sequence Visualization (With Bounding Boxes)

This notebook processes raw 3D point clouds from the labeled dataset, actively converts them into our highly-efficient 2.5D Foveated Grid structure, and visualizes the sequence as an interactive animation.

**Update**: Dynamic and Static objects are now detected, clustered into bounding boxes, and explicitly labeled (e.g., 'Car', 'Pedestrian', 'Guardrail') to vastly improve visual clarity. Terrain remains represented by foveated cells.

In [1]:
import sys
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection
from matplotlib.lines import Line2D
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Add src to path so we can import our pipeline modules
sys.path.insert(0, os.path.abspath('../src'))
from foveated.config import GridConfig, CLASS_KIND
from foveated.io_pcd import load_pcd
from foveated.preprocess import clean
from foveated.grid import build
from foveated.semantics import terrain_analysis, extract_instances

In [2]:
data_dir = '../data/labeled/road_label_test'
pcd_files = sorted(glob.glob(os.path.join(data_dir, '*.pcd')))
print(f'Found {len(pcd_files)} frames in {data_dir}.')

cfg = GridConfig()

def process_frame(filepath):
    raw = load_pcd(filepath)
    cf = clean(raw, cfg)
    g = build(cf, cfg) # This creates our 2.5D FoveatedGrid!
    return g

print('Converting all 3D Point Clouds to 2.5D Foveated Grids...')
grids = []
# Process first 20 frames for animation speed, change to pcd_files to do all
for fp in pcd_files[:20]:
    grids.append(process_frame(fp))
print('Successfully converted to 2.5D!')

Found 41 frames in ../data/labeled/road_label_test.
Converting all 3D Point Clouds to 2.5D Foveated Grids...
Successfully converted to 2.5D!


In [3]:
fig, ax = plt.subplots(figsize=(10, 10), dpi=120)

# Color map matching our project presentation aesthetic
c_map = {
    0: '#6e6e76', # Unknown/Others (Gray)
    1: '#00BFFF', # Terrain (Cyan)
    2: '#9932CC', # Static (Purple)
    3: '#ff4d4d'  # Dynamic (Red)
}

def update(frame_idx):
    ax.clear()
    g = grids[frame_idx]
    
    # Run instance extraction to get objects and bounding boxes
    terrain_analysis(g)
    instances = extract_instances(g)
    
    # Map raw class IDs to our 4 Kinds (Dynamic, Static, Terrain, Unknown)
    kinds = np.vectorize(lambda c: CLASS_KIND.get(c, 0))(g.class_id.astype(int))
    
    patches_by_kind = {0: [], 1: []}
    
    # Create accurate 2.5D cell rectangles ONLY for Terrain and Unknown
    for i in range(g.n_cells):
        k = kinds[i]
        if k in [0, 1]:
            r = g.res[i]
            # g.cy -> X axis, g.cx -> Y axis (to match bird's eye view standard)
            rect = Rectangle((g.cy[i] - r/2, g.cx[i] - r/2), r, r)
            patches_by_kind[k].append(rect)
        
    # Render background cells rapidly using PatchCollection
    for k in [1, 0]: 
        if patches_by_kind[k]:
            pc = PatchCollection(patches_by_kind[k], facecolor=c_map[k], edgecolor='none', alpha=0.6)
            ax.add_collection(pc)
            
    # Render Dynamic/Static objects as bounding boxes instead of individual cells
    for inst in instances:
        k = inst.kind
        color = c_map.get(k, '#000000')
        # Create bounding box
        rect = Rectangle((inst.cy - inst.extent_y/2, inst.cx - inst.extent_x/2), 
                         inst.extent_y, inst.extent_x, 
                         facecolor=color, edgecolor='white', alpha=0.8, lw=1.5)
        ax.add_patch(rect)
        
        # Add text label in center
        ax.text(inst.cy, inst.cx, inst.class_name, color='white', 
                ha='center', va='center', fontsize=9, fontweight='bold',
                bbox=dict(facecolor=color, edgecolor='white', alpha=0.9, pad=0.3, boxstyle='round'))
            
    ax.set_xlim(-40, 40)
    ax.set_ylim(-40, 40)
    ax.set_aspect('equal')
    
    # Aesthetics
    ax.set_facecolor('#f8f9fa')
    fig.patch.set_facecolor('#f8f9fa')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"2.5D Foveated Grid View - Frame {frame_idx+1:02d}", fontsize=18, fontweight='bold', pad=20)
    
    legend_elements = [
        Line2D([0], [0], marker='s', color='w', label='Dynamic (Red)', markerfacecolor=c_map[3], markersize=14),
        Line2D([0], [0], marker='s', color='w', label='Static (Purple)', markerfacecolor=c_map[2], markersize=14),
        Line2D([0], [0], marker='s', color='w', label='Terrain (Cyan)', markerfacecolor=c_map[1], markersize=14),
        Line2D([0], [0], marker='s', color='w', label='Others (Gray)', markerfacecolor=c_map[0], markersize=14)
    ]
    ax.legend(handles=legend_elements, loc='upper right', frameon=True, facecolor='white', edgecolor='lightgray')

print('Generating animation... (This may take a moment to render)')
anim = FuncAnimation(fig, update, frames=len(grids), interval=200)
plt.close() # Prevent static plot
HTML(anim.to_jshtml())


Generating animation... (This may take a moment to render)
